# The planning problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/05_problem.ipynb)

`PlanningProblem`: the task every planner, evaluator and learner reads, deterministic or stochastic, continuous or on a clock.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A problem is a system, its sets, a cost and a horizon

A `PlanningProblem` states the task, never how to solve it:

$$\min J = \int_{t_0}^{t_0 + t_f} e^{-\rho (t - t_0)} g\, dt + e^{-\rho t_f} h(x(t_f))
\quad \text{s.t.} \quad \dot x = f(x, u, t),\; x(t) \in X,\; u(t) \in U,\;
x(t_0) \in X_0,\; x(t_0 + t_f) \in X_f.$$

Every planner, controller evaluation and learner reads the same object. Here is the swing-up
of a pendulum: start at rest at the bottom, end near the upright inside the state and torque
limits, within five seconds.

In [ ]:
import numpy as np

from minilink import BallSet, Pendulum, PlanningProblem, QuadraticCost

plant = Pendulum()
UP = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(
    plant, Q=np.diag([1.0, 0.1]), R=0.01 * np.eye(1), xbar=UP
)

problem = PlanningProblem(
    plant,
    x_start=np.array([0.0, 0.0]),
    cost=cost,
    X=plant.state.box,
    U=plant.input_box,
    Xf=BallSet(UP, 0.2),
    tf=5.0,
)
problem

## 2. The fields and what they mean

| Field | Meaning | Default |
| --- | --- | --- |
| `sys` | the model, any `System` | required |
| `X`, `U` | hard state and input constraints | unconstrained |
| `X0` | the start constraint $x(t_0) \in X_0$, what planners enforce | from `x_start` |
| `x_start` | where runs start: a point, or a law | from `X0`, else `sys.x0` |
| `Xf` | the terminal constraint; `None` is a free end | `None` |
| `cost` | the `CostFunction` | `None` (feasibility only) |
| `tf` | the horizon in seconds; `None` or `inf` is infinite | `None` |
| `t_start`, `discount_rate` | the horizon's start $t_0$ and the discount rate $\rho$ | `0`, `0` |

A few derived answers: `goal_point()` is the point a heuristic aims at (`Xf`'s point, else
the cost's target), `horizon_kind()` is `"finite"` or `"infinite"`, and
`discount_factor(dt)` is $\gamma = e^{-\rho\, \Delta t}$ for a solver on a clock.

In [ ]:
problem.X0, problem.goal_point(), problem.horizon_kind()

A problem is immutable in spirit: `replace(**changes)` returns a new problem with some
fields changed, the rest shared. The infinite-horizon, discounted version of the same task,
the form value iteration and LQR expect:

In [ ]:
infinite_problem = problem.replace(tf=None, Xf=None, discount_rate=0.5)
infinite_problem.horizon_kind(), infinite_problem.discount_factor(0.05)

## 3. Stochastic problems: a law where a number goes

A problem becomes stochastic by its parts:

- `x_start=` a `Distribution`: the start is drawn per run, and `X0` becomes its support;
- `params=` laws over the parameters, in the tree's own shape (domain randomization);
- `disturbances=` source blocks on the plant's input ports (a disturbance has a time
  behaviour, so it is a block, never a bare law).

`is_stochastic` is read off the parts, `drawn_paths()` names the drawn leaves,
`sample_x0` draws starts, `realize(key, n)` draws everything into one tree, and `nominal()`
is the problem with every law at its mean.

In [ ]:
from minilink import Uniform

random_problem = PlanningProblem(
    plant,
    x_start=Uniform([-0.2, -0.2], [0.2, 0.2]),
    cost=cost,
    tf=5.0,
    params={"plant": {"m": Uniform([0.8], [1.2])}},
)
random_problem.is_stochastic, random_problem.drawn_paths(), random_problem.X0

In [ ]:
random_problem.sample_x0(0, 3), random_problem.nominal().x_start

A disturbance is a source on one of the plant's ports. `NoisyPendulum` has a torque
disturbance port `d` and a measurement noise port `n` beside its command `u`; the problem
wires a `WhiteNoise` block onto `d`, and its seed becomes a drawn leaf, one per run.

In [ ]:
from minilink import NoisyPendulum, WhiteNoise

noisy_plant = NoisyPendulum()
noisy_cost = QuadraticCost.from_system(
    noisy_plant, Q=np.eye(2), R=np.diag([1.0, 0.0, 0.0])
)

noisy_problem = PlanningProblem(
    noisy_plant,
    x_start=np.zeros(2),
    cost=noisy_cost,
    tf=5.0,
    disturbances={"d": WhiteNoise(1, psd=0.04, sample_period=0.01, seed=1)},
)
(
    list(noisy_plant.inputs),
    noisy_problem.drawn_paths(),
    noisy_problem.realize(0, 3)["disturbance"],
)

## 4. The step face: the problem on a clock

Solvers that work tick by tick (value iteration, reinforcement learning, a judge) read the
problem through one face. `problem.discretize(dt)` is the explicit discretization, a
`StepPlanningProblem` with $N = t_f / \Delta t$ ticks, $\gamma = e^{-\rho \Delta t}$, its
plant a `SampledSystem` and its cost a `SampledCost`. `problem.compile(dt=dt)`, with no
controller, gives its face directly: a `StepFace` with

$$(x_{k+1}, c_k) = \mathrm{transition}(x_k, u_k, k), \qquad J = \sum_{k<N} \gamma^k c_k + \gamma^N h(x_N).$$

In [ ]:
step_problem = problem.discretize(0.05)
(
    type(step_problem.sys).__name__,
    type(step_problem.cost).__name__,
    step_problem.N,
    step_problem.gamma,
)

In [ ]:
face = problem.compile(dt=0.05)

x = np.zeros(2)
u = np.array([1.0])
face.transition(x, u, 0), step_problem.compile().transition(x, u, 0)

`face.run(policy)` rolls a policy $u_k = \pi(x_k, k)$ over the $N$ ticks and returns the
states, the inputs, the stage costs and $J$. A constant torque does not swing the pendulum
up, and its $J$ says so; the next notebooks score real controllers.

In [ ]:
xs, us, cs, J = face.run(lambda x, k: np.array([2.0]))
xs.shape, J

Next, [`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb): compile a controller on
the problem and score it, once or over many draws.